# PNAD Income Analysis Pipeline

This notebook is the single executable scientific analysis for the project. Numerical procedures are implemented in `pnad_income`; the notebook orchestrates the pipeline, displays the complete analysis, and persists reproducible products under `outputs/`.


## 1. Configuration

The refined annual records are read from `dados_refined/`. Output products are written to `outputs/figures/`, `outputs/tables/`, and `outputs/reports/`.


In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import (
    plot_ccdf, plot_ccdf_grid, plot_gini_evolution,
    plot_histogram, plot_histogram_grid,
    plot_lorenz_curve, plot_lorenz_grid,
    plot_measure_comparison, plot_measure_comparison_grid,
)
from pnad_income.outputs import build_diagnostics, export_analysis_outputs, prepare_output_paths

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATABASE_PATH = Path(os.environ.get("PNAD_DATABASE_PATH", "../dados_refined")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("PNAD_OUTPUT_PATH", "../outputs")).expanduser()
OUTPUT_PATHS = prepare_output_paths(OUTPUT_ROOT)

CONFIG = PipelineConfig(database_path=DATABASE_PATH, ccdf_base=1.05, start_year=1976, end_year=2025)
SELECTED_YEAR = 2025
SELECTED_YEARS = [1976, 1990, 2001, 2010, 2020, 2025]
GRID_NROWS = 2
GRID_NCOLS = 3

CONFIG


## 2. Database loading and analytical coverage


In [ ]:
results = run_pipeline(CONFIG)
overview = pipeline_overview(results)
display(overview)
display(results.panel.head())


## 3. Annual descriptive statistics


In [ ]:
summary = results.summary
display(summary)


## 4. Plot-selection interface


In [ ]:
plot_histogram(results.panel, year=SELECTED_YEAR, value_col="income", bins=60, yscale="log")
plt.show()

for fig in plot_histogram_grid(
    results.panel, value_col="income", years=SELECTED_YEARS,
    bins=60, yscale="log", nrows=GRID_NROWS, ncols=GRID_NCOLS,
):
    plt.show()


## 5. Annual Gini coefficient


In [ ]:
plot_gini_evolution(summary, value_col="income")
plt.show()


## 6. Annual income histograms — linear frequency scale


In [ ]:
for fig in plot_histogram_grid(
    results.panel, value_col="income", years=results.years,
    bins=60, yscale="linear", nrows=6, ncols=4,
):
    plt.show()


## 7. Annual income histograms — logarithmic frequency scale


In [ ]:
for fig in plot_histogram_grid(
    results.panel, value_col="income", years=results.years,
    bins=60, yscale="log", nrows=6, ncols=4,
):
    plt.show()


## 8. Complementary cumulative distribution function

For nonnegative income \(X\),
\[
\widehat{\overline F}(x)=\frac{1}{N}\sum_{i=1}^N \mathbf{1}(X_i\ge x).
\]
Finite zero-income observations remain in the denominator.


In [ ]:
ccdf = results.ccdf_nominal_adjusted
display(ccdf.head(20))


## 9. Individual annual distribution


In [ ]:
plot_ccdf(ccdf, year=SELECTED_YEAR, measure="income", transform="loglog")
plt.show()


## 10. Annual CCDFs — linear axes


In [ ]:
for fig in plot_ccdf_grid(
    ccdf, measure="income", years=results.years,
    transform="linear", nrows=6, ncols=4,
):
    plt.show()


## 11. Annual CCDFs — log-log axes


In [ ]:
for fig in plot_ccdf_grid(
    ccdf, measure="income", years=results.years,
    transform="loglog", nrows=6, ncols=4,
):
    plt.show()


## 12. Legacy double-log diagnostic


In [ ]:
for fig in plot_ccdf_grid(
    ccdf, measure="income", years=results.years,
    transform="double_log", nrows=6, ncols=4,
):
    plt.show()


## 13. Annual Lorenz curves


In [ ]:
for fig in plot_lorenz_grid(
    results.panel, value_col="income", years=results.years,
    nrows=6, ncols=4,
):
    plt.show()


## 14. Individual Lorenz curve


In [ ]:
plot_lorenz_curve(results.panel, year=SELECTED_YEAR, value_col="income")
plt.show()


## 15. Nominal versus adjusted distributions


In [ ]:
for fig in plot_measure_comparison_grid(
    ccdf, measures=("income", "income_adj"), years=results.years,
    transform="loglog", nrows=6, ncols=4,
):
    plt.show()

plot_measure_comparison(
    ccdf, year=SELECTED_YEAR,
    measures=("income", "income_adj"), transform="loglog",
)
plt.show()


## 16. Effective-income availability


In [ ]:
ccdf_effective = results.ccdf_habitual_effective
if ccdf_effective.empty:
    print("The current refined database does not contain usable income_effective observations.")
else:
    display(ccdf_effective.head(20))


## 17. Final data-quality diagnostics


In [ ]:
diagnostics = build_diagnostics(results)
display(diagnostics)


## 18. Persist all scientific outputs

The following call writes the complete reproducible output set under `outputs/`. It saves all standard tables, every page of complete-series figures, the selected-year examples, the selected-year grids, and an `outputs/manifest.csv` inventory.


In [ ]:
manifest = export_analysis_outputs(
    results,
    output_root=OUTPUT_ROOT,
    selected_year=SELECTED_YEAR,
    selected_years=SELECTED_YEARS,
    grid_nrows=GRID_NROWS,
    grid_ncols=GRID_NCOLS,
    complete_nrows=6,
    complete_ncols=4,
    histogram_bins=60,
    dpi=200,
)
display(manifest)
print(f"Outputs saved under: {OUTPUT_PATHS.root}")
